In [1]:
import torch 

from importnb import imports
with imports("ipynb"):
    from GPTModel import GPTModel

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])
tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])
Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[ 5.8380e-01, -1.0901e-01, -1.0397e+00,  ...,  1.5072e-02,
          -2.1667e-01, -4.7530e-01],
         [ 2.0323e-01, -2.8204e-01, -8.5981e-01,  ..., -1.9883e-01,
          -8.5702e-04, -3.5104e-01],
         [ 9.0067e-01,  1.5839e-01, -6.5688e-01,  ...,  2.4166e-02,
          -9.1903e-01, -1.8686e-01],
         [-1.1552e+00,  8.3347e-01, -4.3708e-01,  ...,  8.2582e-01,
           2.1277e-01, -4.4075e-01]],

        [[ 2.0957e-02, -4.8953e-01, -5.8782e-01,  ...,  3.3615e-01,
           1.2045e-02, -7.8100e-01],
         [ 1.0735e-01, -3.4241e-02, -1.5861e-01,  ...,  4.1621e-01,
          -2.7504e-01,  3.1481e-01],
         [ 1.1170e+00,  7.1822e-01,  2.7271e-02,  ...,  5.3295e-01,
          -7.6368e-02,  9.0427e-

In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257, 
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (attn): MultiHeadAttention(
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (heads): ModuleList(
          (0-11): 12 x CasualAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (W_key): Linear(in_features=768, out_features=64, bias=False)
            (W_query): Linear(in_features=768, out_features=64, bias=False)
            (W_value): Linear(in_features=768, out_features=64, bias=False)
          )
        )
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False

## Generate text using GPT

In [3]:
# Utility funcitons for text to token ID conversion 
import tiktoken 
from GPTModel import generate_text_simple 

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

In [4]:
start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_token=10,
    context_size=GPT_CONFIG_124M["context_length"]
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you646Septemberadvمブ punched dub Jin MLS wave


## Calculating the text generation loss

In [5]:
inputs = torch.tensor([[16833, 3626, 6100], [40,1107, 588]])
targets = torch.tensor([[3626, 6100, 345 ], [1107, 588, 11311]])

In [6]:
# 0. Apply softmax
with torch.no_grad():
    logits = model(inputs)
probas = torch.softmax(logits, dim=-1)
print(probas.shape)

torch.Size([2, 3, 50257])


In [7]:
# 1. Determining the next token by determining the token with largest probability
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("Token IDs:\n", token_ids)

Token IDs:
 tensor([[[46104],
         [16031],
         [13520]],

        [[25050],
         [22809],
         [32271]]])


In [8]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

Targets batch 1:  effort moves you
Outputs batch 1:  Veil saves winds


In [9]:
# 2. The three target token ID probabilities for each batch are
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 1:", target_probas_1)

text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 2:", target_probas_2)

Text 1: tensor([3.4224e-05, 2.6318e-05, 1.1486e-05])
Text 2: tensor([1.2981e-05, 6.0671e-05, 6.7075e-06])


In [10]:
# 3.Log probabilities 
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)

# Average it 
avg_log_probas = torch.mean(log_probas)
avg_log_probas = avg_log_probas * -1
print("Negative average log proba: ", avg_log_probas)

tensor([-10.2826, -10.5452, -11.3744, -11.2521,  -9.7100, -11.9123])
Negative average log proba:  tensor(10.8461)


In [11]:
# Do 0 to 3 in one step 
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

# pytorch inbuilt
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print("Loss: ", loss)

Flattened logits: torch.Size([6, 50257])
Flattened targets: torch.Size([6])
Loss:  tensor(10.8461)


In [12]:
# Perplexity metrics
perplexity = torch.exp(loss)
print(f"Perplexity loss: {perplexity}")
# In the given example, this would translate to the model being unsure about which among
# 51333 tokens in the vocabulary to generate as the next token

Perplexity loss: 51333.55078125


## Calculating training and validation set losses

In [13]:
file_path = "../../Dataset/the-verdict.txt"
with open(file_path, "r", encoding="utf-8") as file:
    text_data = file.read()

train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

In [14]:
# Dataset related utilities
from torch.utils.data import Dataset, DataLoader
class GPTDatasetV1(Dataset):
    ''' 
    This creates dataset, i.e, generates input output pairs from given text.
    This will be used in creating LLMs
    '''
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
        
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

# A data loader to generate batches of input output pairs from dataset
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    dataloader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataloader

In [15]:
train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)
val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [16]:
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)
print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

Train loader:
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])

Validation loader:
torch.Size([2, 256]) torch.Size([2, 256])


In [17]:
def calc_loss_batch(input_batch, target_batch, model, device):
    ''' 
    Calculate the loss between the target and predicted with device support
    '''
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss

In [18]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    ''' 
    Function to compute the training and validation loss using data loader
    '''
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduces no of batche to match total number of batches in data loader if num_batches increase no of batches in dataloader
        num_batches = min(num_batches, len(data_loader))
    
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch=input_batch, target_batch=target_batch, model = model, device=device)
            total_loss += loss.item()
        else:
            break
    return total_loss/num_batches

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)
print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 10.981420940823025
Validation loss: 10.973359107971191


In [20]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [21]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple( model=model, idx=encoded, max_new_token=50, context_size=context_size )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))
    model.train()

In [22]:
def train_model_simple(model, train_loader, val_loader,
    optimizer, device, num_epochs,
    eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()

            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, track_tokens_seen

In [23]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr = 0.0004, weight_decay=0.1)
num_epochs = 10

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

Ep 1 (Step 000000): Train loss 9.780, Val loss 9.946
Ep 1 (Step 000005): Train loss 8.119, Val loss 8.353
Every effort moves you,,,,,,,,,,,,,,.                                   
Ep 2 (Step 000010): Train loss 6.658, Val loss 7.051
Ep 2 (Step 000015): Train loss 5.944, Val loss 6.621
Every effort moves you, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and
Ep 3 (Step 000020): Train loss 13.914, Val loss 14.128
Ep 3 (Step 000025): Train loss 5.622, Val loss 6.464
Every effort moves you, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and, and
Ep 4 (Step 000030): Train loss 5.340, Val loss 6.417
Ep 4 (Step 000035): Train loss 4.955, Val loss 6.342
Every effort moves you, and, and a--I--I--I was a of the of the of the of the of the of the of the of the of the of the of the of the of the of the of the of the of theisburn,
Ep 5 (Step 000040): Train los

In [5]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax2 = ax1.twiny()
    ax2.plot(tokens_seen, train_losses, alpha=0)
    ax2.set_xlabel("Tokens seen")
    fig.tight_layout()
    plt.show()

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

NameError: name 'train_losses' is not defined